# glcuda diagnostics — glbench only

No llama.cpp. Nothing here needs it, and building it on Kaggle costs 28
minutes and has run out of memory overnight.

This notebook answers three questions about **glcuda's own** performance, all
of which need a real GPU that the development machine does not have:

| # | Question | How |
|---|---|---|
| 1 | Does the disabled r256 GEMM pass parity on hardware? | `cargo test -p glcuda` |
| 2 | Where does prefill time actually go? | `GLCUDA_PROFILE_PREFILL=1` |
| 3 | How much of decode is host-side sampling, not GPU work? | `GLCUDA_PROFILE_DECODE=1` |

### Why each one matters

**(1) A measured 41–43% speedup is switched off.** `glcuda/src/runner.rs`
drives the tensor-core GEMM in 64-row sub-slabs, so a 220-token prompt
re-reads every weight four times. `gl_gemm_mma_q8_r256` (256-row reuse) is
written, shipped in the PTX, and benchmarked 41–43% faster with 96 registers
and no spill — but wiring it produced `CUDA_ERROR_MISALIGNED_ADDRESS` and it
was reverted until its parity test goes green *on hardware*. The comment
records why that never happened: **"the T4 runs only bench + profiler, not
cargo test."** This notebook runs cargo test.

⛔ `glcuda/tests/parity.rs` **skips** rather than fails when no CUDA device is
present, printing `SKIP: no CUDA driver/device on this machine`. A green
summary can therefore mean "nothing touched a GPU" — which is exactly the
state the blocker is already in. The cell below reports a skip as proving
nothing, never as a pass.

**(2) Two prefill candidates, deliberately unranked.** An audit found
attention running `attn_decode_rows` (a decode-shaped kernel applied per
prefill row, with no tensor-core path) and no kernel fusion anywhere
(`rms_norm → quantize → gemm` as three kernels, four `quantize_q8` passes per
layer, roughly 45 launches per layer). A 2026-07-12 profile put the attention
core at 39% of prefill — but that was a different session and a different
model. **This notebook does not rank them.** The bucket split decides.

**(3) glbench's decode number includes work llama.cpp's does not.** Every
token copies full-vocabulary logits to the host, applies a repetition penalty
and samples on the CPU, serialised between graph launches with the GPU idle.
`GLCUDA_PROFILE_DECODE=1` splits GPU from host so the engine's decode rate can
be separated from the sampling pipeline's.

### On reading the profiled runs

Profile mode syncs at phase boundaries. The **totals are inflated by the act
of measuring**; the **split between buckets is the usable result**. The
unperturbed reference numbers come from the plain run in Step 4.

## Step 1 — Config

In [ ]:
# ---- edit these if needed -------------------------------------------------
REPO_URL = "https://github.com/gwenland-org/gwenland-ai.git"
BRANCH   = "glbench-vs-llamacpp"
GH_TOKEN = ""

MODEL_REPO = "https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct-GGUF/resolve/main"
MODEL_FILE = "qwen2.5-0.5b-instruct-q4_k_m.gguf"

GEN_TOKENS = 128
WARMUP     = 3
ITERS      = 10
# --------------------------------------------------------------------------

import os, sys, re, json, time, glob, shutil, subprocess, urllib.request

WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else "/content"
os.makedirs(WORK, exist_ok=True)
REPO_DIR = os.path.join(WORK, "gwenland-ai")
OUT_DIR  = WORK

def sh(cmd, cwd=None, timeout=7200, env=None):
    # stdin=DEVNULL so nothing can sit waiting for a human that is not there.
    e = dict(os.environ)
    if env:
        e.update(env)
    try:
        p = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True,
                           timeout=timeout, env=e, stdin=subprocess.DEVNULL)
        return p.returncode, p.stdout, p.stderr
    except subprocess.TimeoutExpired:
        return 124, "", f"timed out after {timeout}s"
    except Exception as ex:
        return 125, "", f"{type(ex).__name__}: {ex}"

rc, out, _ = sh(["nvidia-smi", "--query-gpu=name,compute_cap,memory.total",
                 "--format=csv,noheader"], timeout=60)
GPU_LINE = out.strip() if rc == 0 else "no nvidia-smi"
GPU_COUNT = len([ln for ln in GPU_LINE.splitlines() if ln.strip()]) if rc == 0 else 0
print(f"gpus : {GPU_COUNT}")
for ln in GPU_LINE.splitlines():
    print(f"       {ln}")

# Kaggle hands out two T4s. glcuda runs on one; pinning keeps every number in
# this notebook comparable to every other, and to the runs already recorded.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
if GPU_COUNT > 1:
    print(f"\n{GPU_COUNT} GPUs visible -- pinned to device 0 "
          f"(CUDA_VISIBLE_DEVICES=0) so results stay comparable")

RUN_META = {
    "date_utc": time.strftime("%Y-%m-%d %H:%M:%S UTC", time.gmtime()),
    "gpu": GPU_LINE.replace("\n", " | "),
    "gpu_count": GPU_COUNT,
}
if GPU_COUNT == 0:
    print("\nWARNING: no GPU detected. Every diagnostic below needs one; the "
          "parity suite will SKIP and report nothing.")


## Step 2 — Repo and toolchain

In [ ]:
if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    url = REPO_URL.replace("https://", f"https://{GH_TOKEN}@") if GH_TOKEN else REPO_URL
    rc, o, e = sh(["git", "clone", "--depth", "1", "--branch", BRANCH, url, REPO_DIR])
    print((o or e)[-1200:])
else:
    sh(["git", "fetch", "--depth", "1", "origin", BRANCH], cwd=REPO_DIR)
    sh(["git", "checkout", BRANCH], cwd=REPO_DIR)
    sh(["git", "reset", "--hard", f"origin/{BRANCH}"], cwd=REPO_DIR)
    print("refreshed existing clone")

rc, o, _ = sh(["git", "log", "--oneline", "-1"], cwd=REPO_DIR)
GL_COMMIT = o.strip()
print("commit :", GL_COMMIT)

if shutil.which("cargo") is None:
    os.system("curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | "
              "sh -s -- -y --default-toolchain stable --profile minimal")
os.environ["PATH"] = os.path.expanduser("~/.cargo/bin") + ":" + os.environ["PATH"]
rc, o, _ = sh(["cargo", "--version"], timeout=180)
print("cargo  :", o.strip())


## Step 3 — Model

In [ ]:
MODEL_PATH = None
for cand in [os.path.join(WORK, MODEL_FILE), os.path.join(REPO_DIR, MODEL_FILE)]:
    if os.path.exists(cand) and os.path.getsize(cand) > 10_000_000:
        MODEL_PATH = cand
        break

if MODEL_PATH is None and os.path.isdir("/kaggle/input"):
    for root, _, files in os.walk("/kaggle/input"):
        for f in files:
            if f.lower().endswith(".gguf") and "0.5b" in f.lower():
                MODEL_PATH = os.path.join(root, f)
                break
        if MODEL_PATH:
            break

if MODEL_PATH is None:
    dest = os.path.join(WORK, MODEL_FILE)
    print(f"downloading {MODEL_REPO}/{MODEL_FILE}")
    rc, o, e = sh(["curl", "-fL", "--retry", "3", "-o", dest,
                   f"{MODEL_REPO}/{MODEL_FILE}"], timeout=1800)
    if rc == 0 and os.path.exists(dest) and os.path.getsize(dest) > 10_000_000:
        MODEL_PATH = dest
    else:
        print("download failed:", e[-400:])

if not MODEL_PATH:
    raise RuntimeError("no model — nothing below can run")
MODEL_BYTES = os.path.getsize(MODEL_PATH)
print(f"model : {MODEL_PATH}")
print(f"size  : {MODEL_BYTES/1e9:.3f} GB")


## Step 4 — Build glbench, then the reference run

The binary is cached in `/kaggle/working` and keyed to the commit, so a saved
session skips the build next time. The reference run below is **unperturbed**:
no profiling env vars, so its numbers are the ones to quote.

In [ ]:
GL_BIN = os.path.join(REPO_DIR, "target", "release", "glbench")
BIN_CACHE = os.path.join(WORK, "glbench-bin-cache")
CACHE_MANIFEST = os.path.join(BIN_CACHE, "manifest.json")

# Reuse only on an exact commit match: a binary from other code is a binary
# measuring other code.
RESTORED = False
if os.path.exists(CACHE_MANIFEST):
    try:
        man = json.load(open(CACHE_MANIFEST, encoding="utf-8"))
        src = os.path.join(BIN_CACHE, "glbench")
        if man.get("commit") == GL_COMMIT and os.path.exists(src):
            os.makedirs(os.path.dirname(GL_BIN), exist_ok=True)
            shutil.copy2(src, GL_BIN)
            os.chmod(GL_BIN, 0o755)
            RESTORED = True
            print("reusing cached glbench binary (same commit)")
        else:
            print(f"cache is for a different commit ({man.get('commit')}); rebuilding")
    except Exception as ex:
        print(f"cache unreadable ({type(ex).__name__}); rebuilding")

BUILD_SECS = 0.0
if not RESTORED:
    print("building glbench (release)...")
    t0 = time.time()
    rc, o, e = sh(["cargo", "build", "--release", "-p", "glbench"],
                  cwd=REPO_DIR, timeout=7200)
    BUILD_SECS = time.time() - t0
    if rc != 0:
        print((o + e)[-5000:])
        raise RuntimeError("glbench build failed")
    print(f"built in {BUILD_SECS/60:.1f} min")
    os.makedirs(BIN_CACHE, exist_ok=True)
    shutil.copy2(GL_BIN, os.path.join(BIN_CACHE, "glbench"))
    json.dump({"commit": GL_COMMIT}, open(CACHE_MANIFEST, "w", encoding="utf-8"))
    print(f"cached to {BIN_CACHE} — save this notebook version to keep it")


In [ ]:
# The reference measurement. No profiling env vars: profile mode syncs at
# phase boundaries and inflates exactly what it reports.
GL_JSON_PATH = os.path.join(OUT_DIR, "glbench_result.json")
GL_CMD = [GL_BIN, "run",
          "--engine", "glcuda",
          "--model", MODEL_PATH,
          "--tokens", str(GEN_TOKENS),
          "--warmup", str(WARMUP),
          "--iters", str(ITERS),
          "--out", GL_JSON_PATH]
print("$ " + " ".join(GL_CMD) + "\n")

t0 = time.time()
rc, GL_STDOUT, GL_STDERR = sh(GL_CMD, cwd=REPO_DIR, timeout=7200)
GL_WALL = time.time() - t0
open(os.path.join(OUT_DIR, "glbench_output.txt"), "w", encoding="utf-8").write(
    GL_STDOUT + "\n===== stderr =====\n" + GL_STDERR)
print(GL_STDOUT[-6000:])
if rc != 0:
    print("\n===== stderr =====\n" + GL_STDERR[-3000:])
    raise RuntimeError(f"glbench --engine glcuda failed (exit {rc})")

GL_JSON = json.load(open(GL_JSON_PATH, encoding="utf-8"))
GL_AN = GL_JSON.get("analysis") or {}
_its = (GL_JSON.get("measurements") or {}).get("iterations") or []
PROMPT_TOKENS = int(_its[0]["prompt_tokens"]) if _its and _its[0].get("prompt_tokens") else None
GL_PRE = (GL_AN.get("prefill_tps") or {})
GL_DEC = (GL_AN.get("decode_tps") or {})
print(f"\nprompt tokens : {PROMPT_TOKENS}")
print(f"prefill       : {GL_PRE.get('mean')} tok/s (median {GL_PRE.get('median')})")
print(f"decode        : {GL_DEC.get('mean')} tok/s (median {GL_DEC.get('median')})")
print(f"wall          : {GL_WALL/60:.1f} min")


## Step 5 — Question 1: does r256 pass parity on hardware?

⛔ Read the verdict, not the summary line. The suite skips without a device,
so `0 failed` can mean nothing ran.

In [ ]:
print("$ cargo test -p glcuda --release -- --nocapture\n")
rc_t, t_out, t_err = sh(["cargo", "test", "-p", "glcuda", "--release",
                         "--", "--nocapture"],
                        cwd=REPO_DIR, timeout=7200)
T_HAY = (t_out or "") + "\n" + (t_err or "")
open(os.path.join(OUT_DIR, "glcuda_tests.txt"), "w", encoding="utf-8").write(T_HAY)

SKIPPED = "SKIP: no CUDA driver/device" in T_HAY
R256_LINE = None
for line in T_HAY.splitlines():
    if "r256_matches_dequantized_reference" in line:
        R256_LINE = line.strip()

for ln in T_HAY.splitlines():
    if ln.startswith("test result:") or "SKIP:" in ln:
        print(ln)

if SKIPPED:
    R256_VERDICT = "SKIPPED — no CUDA device; this run proves nothing"
elif R256_LINE and " ok" in R256_LINE:
    R256_VERDICT = "PASSED on hardware"
elif R256_LINE and "FAILED" in R256_LINE:
    R256_VERDICT = "FAILED on hardware"
elif R256_LINE:
    R256_VERDICT = f"ran, unclear: {R256_LINE}"
else:
    R256_VERDICT = "test not found in output"

print(f"\ngemm_mma_q8_r256_matches_dequantized_reference: {R256_VERDICT}")
if R256_LINE:
    print(f"  {R256_LINE}")
if SKIPPED:
    print("  ^ skipped for lack of a device. NOT a pass: 'never run on "
          "hardware' is precisely the blocker.")
elif R256_VERDICT == "PASSED on hardware":
    print("  ^ the parity blocker is cleared. What remains is the "
          "CUDA_ERROR_MISALIGNED_ADDRESS seen when wiring it into the real "
          "prefill scratch — a different bug from the one this test covers.")


## Step 6 — Question 2: where does prefill time go?

Buckets come from `GLCUDA_PROFILE_PREFILL=1`. The totals are perturbed by the
syncs that produce them; the split is not.

In [ ]:
print("$ GLCUDA_PROFILE_PREFILL=1 glbench run ...\n")
rc_p, p_out, p_err = sh(GL_CMD[:-1] + [os.path.join(OUT_DIR, "glbench_prefill_profile.json")],
                        cwd=REPO_DIR, timeout=3600,
                        env={"GLCUDA_PROFILE_PREFILL": "1"})
P_HAY = (p_out or "") + "\n" + (p_err or "")
open(os.path.join(OUT_DIR, "glcuda_prefill_profile.txt"), "w", encoding="utf-8").write(P_HAY)

PREFILL_BUCKETS = [ln.strip() for ln in P_HAY.splitlines()
                   if re.search(r"(qkv|attn|ffn|gate|down|elt|core|norm|kv)\b", ln, re.I)
                   and re.search(r"\d", ln) and ("ms" in ln or "%" in ln)]

if PREFILL_BUCKETS:
    print("measured prefill buckets:")
    for b in PREFILL_BUCKETS:
        print("   ", b)
    print("\nThe largest bucket decides what to fix. Not the audit's order.")
else:
    print("no per-bucket lines found — reported as UNMEASURED.")
    print("The two audit candidates stay unranked; an unmeasured guess is not "
          "a priority.")
    print("\n--- tail, for diagnosis ---")
    print(P_HAY[-2500:])


## Step 7 — Question 3: how much of decode is host-side sampling?

glcuda's decode loop copies full-vocabulary logits to the host, applies the
repetition penalty and samples on the CPU — all between graph launches, GPU
idle. `llama-bench`'s generation test does none of that (it feeds a random
next token), so any comparison against it is affected by whatever this cell
measures.

In [ ]:
print("$ GLCUDA_PROFILE_DECODE=1 glbench run ...\n")
rc_d, d_out, d_err = sh(GL_CMD[:-1] + [os.path.join(OUT_DIR, "glbench_decode_profile.json")],
                        cwd=REPO_DIR, timeout=3600,
                        env={"GLCUDA_PROFILE_DECODE": "1"})
D_HAY = (d_out or "") + "\n" + (d_err or "")
open(os.path.join(OUT_DIR, "glcuda_decode_profile.txt"), "w", encoding="utf-8").write(D_HAY)

_m = re.search(r"\[decode split\][^\n]*", D_HAY)
DECODE_SPLIT = _m.group(0) if _m else None
DECODE_GPU_MS = DECODE_HOST_MS = DECODE_HOST_SHARE = None
if DECODE_SPLIT:
    print(DECODE_SPLIT)
    _g = re.search(r"GPU\s*([\d.]+) ms/tok", DECODE_SPLIT)
    _h = re.search(r"HOST\s*([\d.]+) ms/tok", DECODE_SPLIT)
    _s = re.search(r"host share\s*(\d+)%", DECODE_SPLIT)
    DECODE_GPU_MS = float(_g.group(1)) if _g else None
    DECODE_HOST_MS = float(_h.group(1)) if _h else None
    DECODE_HOST_SHARE = int(_s.group(1)) if _s else None
    if DECODE_GPU_MS:
        print(f"\nGPU-only decode rate: {1000.0/DECODE_GPU_MS:.1f} tok/s")
        print(f"reference (unperturbed, full pipeline): {GL_DEC.get('mean')} tok/s")
        print("\nThe first is what a forward-pass-only benchmark measures. The "
              "second is what generating text actually costs. Both are real; "
              "they answer different questions.")
else:
    print("no [decode split] line — reported as UNMEASURED, not estimated.")
    print("\n--- tail, for diagnosis ---")
    print(D_HAY[-2000:])


## Step 8 — Write `GLCUDA_DIAGNOSTICS.md`

In [ ]:
NL = "\n"
B = []
def w(s=""):
    B.append(s)

w("# glcuda diagnostics")
w()
w(f"- **Date (UTC):** {RUN_META['date_utc']}")
w(f"- **GPU:** {RUN_META['gpu']}"
  + (f" — {RUN_META['gpu_count']} present, pinned to device 0" if RUN_META['gpu_count'] > 1 else ""))
w(f"- **Commit:** `{GL_COMMIT}`")
w(f"- **Model:** `{os.path.basename(MODEL_PATH)}`, {MODEL_BYTES/1e9:.3f} GB")
w(f"- **Workload:** {PROMPT_TOKENS} prompt tokens, {GEN_TOKENS} generated, "
  f"{ITERS} iterations, {WARMUP} warmup")
w()
w("## Reference measurement (unperturbed)")
w()
w("| Phase | mean | median | min | max | std | ±95% CI |")
w("|---|---:|---:|---:|---:|---:|---:|")
for name, s in (("prefill", GL_PRE), ("decode", GL_DEC)):
    w("| {} | {} | {} | {} | {} | {} | {} |".format(
        name,
        *[("{:.1f}".format(s[k]) if isinstance(s.get(k), (int, float)) else "—")
          for k in ("mean", "median", "min", "max", "std_dev", "ci95")]))
w()
_eff = GL_AN.get("ceiling_efficiency")
w(f"Bottleneck verdict: `{GL_AN.get('bottleneck')}`"
  + (f", {_eff*100:.0f}% of the bandwidth ceiling." if isinstance(_eff, (int, float))
     else ", no ceiling efficiency reported."))
w()
w("> The bottleneck label is a threshold on the ceiling fraction, not an "
  "observation of kernel launches — `bottleneck::classify` returns "
  "`launch_overhead` for anything under 40%. glcuda already replays a captured "
  "CUDA graph per token, so read that label as \"far from the ceiling\", not "
  "as a diagnosis.")
w()

# ---- Q1 -------------------------------------------------------------------
w("## 1. r256 GEMM parity on hardware")
w()
w(f"The MMA GEMM runs in 64-row sub-slabs, so a {PROMPT_TOKENS}-token prompt "
  f"re-reads every weight four times. `gl_gemm_mma_q8_r256` (256-row reuse) is "
  f"written, is in the shipped PTX, and benchmarks 41–43% faster at kernel "
  f"level. It is disabled pending this test passing on a GPU.")
w()
w(f"**Verdict: {R256_VERDICT}**")
if R256_LINE:
    w()
    w("```")
    w(R256_LINE)
    w("```")
w()
if SKIPPED:
    w("`glcuda/tests/parity.rs` skips rather than fails without a device, so "
      "the suite's green summary here means nothing ran. That is the same "
      "green summary the blocked state already produces.")
elif R256_VERDICT == "PASSED on hardware":
    w("The parity blocker is cleared. What remains is the "
      "`CUDA_ERROR_MISALIGNED_ADDRESS` that appeared when wiring r256 into the "
      "real prefill scratch — a different bug from the one this test covers, "
      "and one the isolated kernel bench cannot reproduce because it feeds "
      "synthetic buffers.")
elif R256_VERDICT == "FAILED on hardware":
    w("The kernel does not match the dequantised reference on this device. "
      "The 41–43% figure describes a kernel that computes the wrong answer, "
      "so it is not a speedup yet.")
w()

# ---- Q2 -------------------------------------------------------------------
w("## 2. Where prefill time goes")
w()
if PREFILL_BUCKETS:
    w("Measured on this model and this device:")
    w()
    w("```")
    for b in PREFILL_BUCKETS:
        w(b)
    w("```")
    w()
    w("Profile mode syncs at phase boundaries, so the totals are inflated by "
      "the measurement. The split is the usable part.")
else:
    w("**Unmeasured.** No per-bucket lines appeared in the profiled run.")
w()
w("The audit's two candidates:")
w()
w("- **Attention** — `attn_decode_rows`, a decode-shaped kernel applied per "
  "prefill row, with no tensor-core path at all. A 2026-07-12 profile put the "
  "attention core at 39% of prefill, on a different session and model.")
w("- **Fusion** — none. `rms_norm → quantize → gemm` are three separate "
  "kernels, there are four `quantize_q8` passes per layer, and roughly 45 "
  "launches per layer (about 1,080 for 24 layers), each re-reading the "
  "activation tensor from global memory.")
w()
w("**Not ranked here.** Which one to write is decided by the split above. "
  "Estimates in this repository have missed by 5.4x and this machine drifts "
  "between sessions, so an audit's discovery order is not evidence.")
w()

# ---- Q3 -------------------------------------------------------------------
w("## 3. Decode: GPU work vs host-side sampling")
w()
w("Every decode token copies full-vocabulary logits to the host, applies the "
  "repetition penalty and samples on the CPU, serialised between graph "
  "launches with the GPU idle.")
w()
if DECODE_SPLIT:
    w("```")
    w(DECODE_SPLIT)
    w("```")
    w()
    if DECODE_GPU_MS and isinstance(GL_DEC.get("mean"), (int, float)):
        w("| | tok/s | what it answers |")
        w("|---|---:|---|")
        w(f"| GPU only | {1000.0/DECODE_GPU_MS:.1f} | what a forward-pass-only "
          f"benchmark measures |")
        w(f"| Full pipeline | {GL_DEC['mean']:.1f} | what generating text "
          f"actually costs |")
        w()
        w("Both are real; they answer different questions. Any comparison "
          "against a benchmark whose generation loop feeds a random next token "
          "— `llama-bench`'s does — is comparing against the first row while "
          "glbench reports the second.")
        w()
    if DECODE_HOST_SHARE is not None:
        w(f"Host share: **{DECODE_HOST_SHARE}%**. "
          + ("Large enough that overlapping sampling with the next token's "
             "graph launch is a real lever, and it is not a kernel change."
             if DECODE_HOST_SHARE >= 20 else
             "Small, so the decode gap is not mostly sampling and kernel work "
             "is the place to look."))
        w()
    w("This run syncs after every token, so its absolute figures are perturbed "
      "by the measurement. The ratio is the usable part; the reference table "
      "at the top is the unperturbed throughput.")
else:
    w("**Unmeasured.** No `[decode split]` line appeared, so the size of the "
      "sampling overhead is not known from this run — not estimated.")
w()

w("## Appendix: raw output")
w()
for title, body in [("glbench — reference run", GL_STDOUT),
                    ("cargo test -p glcuda", T_HAY),
                    ("GLCUDA_PROFILE_PREFILL=1", P_HAY),
                    ("GLCUDA_PROFILE_DECODE=1", D_HAY)]:
    w(f"### {title}")
    w()
    w("```")
    w((body or "(empty)").strip()[:12000])
    w("```")
    w()

MD_PATH = os.path.join(OUT_DIR, "GLCUDA_DIAGNOSTICS.md")
open(MD_PATH, "w", encoding="utf-8").write(NL.join(B))
print(f"wrote {MD_PATH} ({len(NL.join(B))} chars)")


## Step 9 — Summary

In [ ]:
print("=" * 66)
print("glcuda diagnostics")
print("=" * 66)
print(f"gpu            : {RUN_META['gpu']}")
print(f"commit         : {GL_COMMIT}")
print(f"prefill        : {GL_PRE.get('mean')} tok/s")
print(f"decode         : {GL_DEC.get('mean')} tok/s  (full pipeline)")
if DECODE_GPU_MS:
    print(f"decode GPU-only: {1000.0/DECODE_GPU_MS:.1f} tok/s"
          + (f"  (host share {DECODE_HOST_SHARE}%)" if DECODE_HOST_SHARE is not None else ""))
print()
print(f"Q1 r256 parity : {R256_VERDICT}")
print(f"Q2 prefill     : {len(PREFILL_BUCKETS)} bucket lines"
      if PREFILL_BUCKETS else "Q2 prefill     : UNMEASURED")
print(f"Q3 decode split: {'measured' if DECODE_SPLIT else 'UNMEASURED'}")
print()
print("Next step is whichever of Q1/Q2 the numbers above justify — this "
      "notebook deliberately does not choose for you.")
print(f"\nreport: {os.path.join(OUT_DIR, 'GLCUDA_DIAGNOSTICS.md')}")
